[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jiehou-lab/urban-ai/blob/main/notebooks/lab3_time_series.ipynb)

# Lab 3: Time Series Analysis

**Duration:** ~1.0 hour
**TA lead:** Sean
**Course:** Urban AI — AI-Driven Decision Support for Real-World Urban Challenges (MSU AI-Ready Initiative)

## Learning goals
- Decompose an hourly urban sensor series into trend, seasonality, and residual.
- Fit a simple forecasting model and evaluate it against held-out data.
- See a forecast fail when the real world undergoes a sudden regime change.
- Produce a trend brief with an explicit forecast caveat.


## Before you start: Track A vs. Track B

Every Urban AI lab has two tracks. Both produce the **same artifact**: **Trend brief + forecast caveat**.

- **Track A — No code (default).** Use a chart tool to filter a traffic/AQI series and spot the trend. No installation, no Python required — use this track if you would rather click through a web tool.
- **Track B — Colab (this notebook).** Decompose trend/seasonality and build a simple forecast, changing the horizon. You will run pre-written cells and change only the parameters marked `# ▶ CHANGE ME`. You will never need to write code from scratch.

Both tracks end with the same 4 reflection prompts (the last cell of this notebook).


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

RNG = np.random.default_rng(3)
plt.rcParams["font.size"] = 12
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
print("Setup complete.")

## 2. Load the data: hourly vehicle counts

> **Synthetic-but-realistic data.** The dataset below is generated in this notebook with a fixed
> random seed so the lab runs the same way for everyone, completely offline. It is built to look and
> behave like real urban data, but it is not real. To swap in real data for your own city, instructors
> can replace the data-generation cell with a download/load from a real source such as:
- State/city Department of Transportation traffic-count sensor APIs or open data portals
- EPA Air Quality System (AQS) hourly air-quality monitor data (for an AQI version of this lab)
- City open data portals publishing real-time or historical traffic-sensor feeds


In [ ]:
n_days = 45
hours = pd.date_range("2025-03-01", periods=n_days * 24, freq="h")
day_of_week = hours.dayofweek.to_numpy()
hour_of_day = hours.hour.to_numpy()

daily_pattern = 50 + 40 * np.exp(-((hour_of_day - 8) ** 2) / 8) + 35 * np.exp(-((hour_of_day - 17) ** 2) / 8)
weekend_factor = np.where(day_of_week >= 5, 0.6, 1.0)
trend = np.linspace(0, 15, len(hours))
noise = RNG.normal(0, 6, len(hours))
base_counts = np.asarray((daily_pattern * weekend_factor) + trend + noise)

# Regime change: a road-closure/construction event starts on day 35 and sharply cuts traffic volume.
# A forecasting model trained only on data before this date has no way to know it is coming.
regime_change_start_day = 35
regime_mask = hours >= (hours[0] + pd.Timedelta(days=regime_change_start_day))
counts = base_counts.copy()
counts[regime_mask] = counts[regime_mask] * 0.45
counts = np.clip(counts, 0, None).round().astype(int)

traffic = pd.DataFrame({"timestamp": hours, "vehicle_count": counts})
traffic.head()

### Look at the raw series
Before modeling anything, always look at the plot -- daily rush-hour spikes, weekly weekday/weekend dips, and any obvious break in the pattern should all be visible by eye first.

In [ ]:
fig_raw, ax = plt.subplots(figsize=(12, 4))
ax.plot(traffic["timestamp"], traffic["vehicle_count"])
ax.set_title("Hourly Vehicle Counts (Synthetic Sensor)")
ax.set_xlabel("Date")
ax.set_ylabel("Vehicles per hour")
plt.tight_layout()
plt.show()

### Decompose the series into trend, seasonality, and residual
Separating out the repeating daily pattern from the underlying trend helps a planner tell "this is just rush hour again" apart from "traffic is genuinely growing."

In [ ]:
ts = traffic.set_index("timestamp")["vehicle_count"]
decomposition = seasonal_decompose(ts, period=24, model="additive", extrapolate_trend="freq")

fig_decomp, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
axes[0].plot(decomposition.observed); axes[0].set_title("Observed hourly vehicle counts"); axes[0].set_ylabel("Vehicles/hr")
axes[1].plot(decomposition.trend); axes[1].set_title("Trend component"); axes[1].set_ylabel("Vehicles/hr")
axes[2].plot(decomposition.seasonal); axes[2].set_title("Daily seasonal pattern"); axes[2].set_ylabel("Vehicles/hr")
axes[3].plot(decomposition.resid); axes[3].set_title("Residual (unexplained) component"); axes[3].set_ylabel("Vehicles/hr")
axes[3].set_xlabel("Date")
plt.tight_layout()
plt.show()

### Build a simple forecast
We train the model only on data from before the road closure, then ask it to forecast forward -- exactly what you would do in practice before an event you don't yet know about.

In [ ]:
# ▶ CHANGE ME: which day to cut training data off at
TRAIN_CUTOFF_DAY = 35

train_ts = ts[ts.index < ts.index[0] + pd.Timedelta(days=TRAIN_CUTOFF_DAY)]
test_ts = ts[ts.index >= ts.index[0] + pd.Timedelta(days=TRAIN_CUTOFF_DAY)]

model = ExponentialSmoothing(train_ts, trend="add", seasonal="add", seasonal_periods=24)
fit = model.fit()
forecast = fit.forecast(len(test_ts))
print(f"Trained on {len(train_ts)} hours, forecasting {len(test_ts)} hours ahead.")

### Responsible AI check: watch the forecast fail
The road closure (regime change) begins right where the training data ends. The model has no idea it is coming, so it simply projects the old pattern forward -- and gets it badly wrong.

In [ ]:
fig_forecast, ax = plt.subplots(figsize=(12, 4))
ax.plot(train_ts.index, train_ts.values, label="Training data (before closure)")
ax.plot(test_ts.index, test_ts.values, label="Actual (during closure)", color="black")
ax.plot(test_ts.index, forecast.values, label="Forecast (assumes no closure)", color="red", linestyle="--")
ax.axvline(test_ts.index[0], color="gray", linestyle=":")
ax.set_title("Forecast Failure During a Regime Change (Road Closure)")
ax.set_xlabel("Date")
ax.set_ylabel("Vehicles per hour")
ax.legend()
plt.tight_layout()
plt.show()

mae = float(np.mean(np.abs(forecast.values - test_ts.values)))
mape = float(np.mean(np.abs((forecast.values - test_ts.values) / np.maximum(test_ts.values, 1))) * 100)
print(f"Forecast error during the closure period: MAE = {mae:.1f} vehicles/hr, MAPE = {mape:.1f}%")

## Experiment

Try changing the parameters marked `# ▶ CHANGE ME` in the next cell(s) and re-run. Specifically, try:

1. Move `TRAIN_CUTOFF_DAY` to 40 (a few days into the closure) and re-run -- does the forecast improve?
2. Move `TRAIN_CUTOFF_DAY` to 20 (well before the closure) and re-run -- does the error get worse?
3. Look at the MAPE value each time -- how large an error would you consider "too risky to act on"?


## Artifact: trend brief + forecast caveat

In [ ]:
trend_vals = decomposition.trend.dropna()
trend_change_pct = (trend_vals.iloc[-1] - trend_vals.iloc[0]) / trend_vals.iloc[0] * 100
volume_drop_pct = 100 - (test_ts.mean() / train_ts.mean() * 100)

brief = f'''TREND BRIEF: Hourly Vehicle Counts
- Data window: {traffic["timestamp"].min()} to {traffic["timestamp"].max()}
- Underlying trend changed by approximately {trend_change_pct:.1f}% over the observed (pre-closure) period.
- Strong daily seasonality: two peaks corresponding to morning and evening rush hours.

FORECAST CAVEAT:
A statistical forecast trained on data through day {TRAIN_CUTOFF_DAY} failed to predict the
{volume_drop_pct:.0f}% volume drop caused by the road closure that began on day {regime_change_start_day}.
Mean absolute error during the closure window was {mae:.1f} vehicles/hour ({mape:.1f}% MAPE).
Forecasts assume recent patterns continue; they cannot anticipate unannounced regime changes
(closures, major events, policy changes) unless a human tells the model about them in advance.
'''
print(brief)
with open("lab3_trend_brief.txt", "w") as f:
    f.write(brief)
print("Saved artifact: lab3_trend_brief.txt")

## Reflect (answer in your own words — this is part of your mini-task)

1. What did the tool assume?
2. Who is missing from this data?
3. What would change your recommendation?
4. What must a human verify before this is used?
